Clase 006 · `006_cadenas_listas` — Practicas en serio la manipulación de
cadenas: caracteres invisibles, partir y reordenar una fecha, convertir tipos
con `int()` y `float()`, unir con `join` y terminar con un mini-ETL sobre
datos tipo ESOLMET.

## Texto multilínea con triple comilla

La triple comilla (`"""`) permite saltos de línea dentro de la cadena;
`\t` es un tabulador.

In [5]:
texto_largo = """
este es\tun texto
de varias lineas
y con diferentes caracteres
"""

La celda siguiente **falla a propósito**: con comillas sencillas la cadena no
puede saltar de línea. Compara el mensaje (`SyntaxError`) con la celda
anterior.

In [2]:
texto_mas_largo = "
este es un intento
de texto
"

SyntaxError: unterminated string literal (detected at line 1) (2090748693.py, line 1)

## Eco vs `print`

El eco de la celda muestra la *representación* de la cadena (con `\n` y `\t`
visibles); `print` la muestra ya interpretada.

In [6]:
texto_largo

'\neste es\tun texto\nde varias lineas\ny con diferentes caracteres\n'

In [7]:
print(texto_largo)


este es	un texto
de varias lineas
y con diferentes caracteres



## `split` y los caracteres invisibles

`variables` mezcla espacios múltiples y un tabulador. Partir por `" "` produce
elementos vacíos y pedazos pegados; para diagnosticarlo hay que *ver* los
caracteres invisibles.

In [10]:
variables = "esta    es\tuna\tprueba"

In [12]:
variables.split(" ")

['esta', '', '', '', 'es\tuna\tprueba']

In [11]:
print(variables)

esta    es	una	prueba


::: {.callout-tip}
`repr()` te enseña la cadena como la ve Python, con `\t` y `\n` explícitos.
Es tu lupa cuando un `split` no da lo que esperabas.
:::

In [14]:
repr(variables)

"'esta    es\\tuna\\tprueba'"

In [15]:
print(repr(variables))

'esta    es\tuna\tprueba'


`split()` **sin argumento** parte por cualquier bloque de espacios en blanco
(espacios, tabuladores, saltos de línea). Casi siempre es lo que quieres.

In [17]:
variables.split()

['esta', 'es', 'una', 'prueba']

## Reordenar una fecha con `split`

Objetivo: pasar de `"03/marzo/1978"` (DD/mes/YYYY) a `YYYY/mes/DD`. Primero a
lo bruto, partiendo la cadena tres veces dentro de la f-string…

In [18]:
fecha = "03/marzo/1978" # YYYY/mm/dd

In [20]:
fecha.split("/")

['03', 'marzo', '1978']

In [28]:
print(f"{fecha.split("/")[2]}/{fecha.split("/")[1]}/{fecha.split("/")[0]}")

1978/marzo/03


In [40]:
print(f"{fecha.split("/")[2]}/{fecha.split("/")[1]}/{fecha.split("/")[0]}")

1978/marzo/03


…y mejor así: guardas el resultado de `split` **una vez** en `fecha_s` y lo
reutilizas. Mismo resultado, código más claro y más rápido.

In [41]:
fecha_s = fecha.split("/")
fecha_s

['03', 'marzo', '1978']

In [46]:
print(f"{fecha_s[2]}/{fecha_s[1]}/{fecha_s[0]}")

1978/marzo/03


## `strip` y `replace`, de nuevo

In [48]:
nombres= " hugo, paco, luisa "
nombres.strip()

'hugo, paco, luisa'

In [49]:
nombres.replace("hugo","HUGO")

' HUGO, paco, luisa '

In [50]:
nombres

' hugo, paco, luisa '

## Convertir tipos: `int()` y `float()`

Lo que sale de `split` siempre son cadenas. Para operar con los pedazos hay
que convertirlos; si la conversión no tiene sentido, Python lanza
`ValueError`.

In [52]:
fecha = "03/marzo/1978" # YYYY/mm/dd
fecha_s = fecha.split("/")
fecha_s

['03', 'marzo', '1978']

In [54]:
int(fecha_s[0])

3

In [55]:
float(fecha_s[2])

1978.0

In [60]:
print(f"{float(fecha_s[2])}/{fecha_s[1]}/{float(fecha_s[0])}")

1978.0/marzo/3.0


Un uso típico: construir nombres de archivo con f-strings a partir de los
pedazos (aunque aquí los `float` le meten `.0` al nombre — fíjate).

In [64]:
archivo = f"nombre_{float(fecha_s[2])}/{fecha_s[1]}/{float(fecha_s[0])}"

In [65]:
archivo

'nombre_1978.0/marzo/3.0'

## `join`: el inverso de `split`

`separador.join(lista)` pega los elementos de una lista usando la cadena como
pegamento. La primera prueba usa `nombre` ("Memo") como separador para que se
vea claro quién pega a quién.

In [66]:
nombre = "Memo"
ape_padre = "Barrios"
ape_madre = "del Valle"

In [70]:
nombre.join(["Hola","mundo"])

'HolaMemomundo'

In [72]:
"-".join(["03","marzo","1978"])

'03-marzo-1978'

In [76]:
fecha = "03/marzo/1978"
fecha = fecha.split("/")
"-".join(fecha)

'03-marzo-1978'

::: {.callout-caution}
Después de `fecha = fecha.split("/")`, `fecha` ya **no es cadena, es lista**.
La celda siguiente falla a propósito: los métodos de cadena (`replace`,
`strip`…) no existen en las listas. `count()` sí existe en ambas, pero cuenta
cosas distintas, y las rebanadas también funcionan en listas.
:::

In [ ]:
fecha.replace()
fecha.strip()  #lstrip,rstrip
fecha.split()
fecha.count()

In [78]:
fecha.count("/")

2

In [79]:
fecha[::2]

'0/az/98'

## `splitlines`

`splitlines()` parte un texto multilínea en una lista de renglones, sin que
tengas que acordarte de `"\n"`.

In [81]:
print(texto_largo)


este es	un texto
de varias lineas
y con diferentes caracteres



In [82]:
texto_largo.splitlines()

['', 'este es\tun texto', 'de varias lineas', 'y con diferentes caracteres']

## Mini-ETL: datos estilo ESOLMET

Un adquisidor entrega los datos como un solo texto: encabezado descriptivo,
nombres de columnas, unidades y luego las filas. Con `splitlines()` y una
rebanada `[3::2]` (desde el renglón 3, de dos en dos) extraes solo las filas
de datos. Este patrón —texto crudo → lista limpia— es el corazón de un ETL.

In [96]:
datos_adquisidor = """
Datos de ESOLMET
de Temixco Morelos, administrados por Jesus Quinones
fecha, hora, temperatura exterior,Iglo,Ibeam,Idiff
mmddaa,hh:mm,oC,w/m2,w/m2,w/m2
20/12/98,5:00 am, 34.2,0,0,0
20/12/98,5:00 am, 34.2,0,0,0
20/12/98,5:00 am, 34.2,0,0,0
20/12/98,5:00 am, 34.2,0,0,0
20/12/98,5:00 am, 34.2,0,0,0
20/12/98,5:00 am, 34.2,0,0,0
20/12/98,5:00 am, 34.2,0,0,0
"""

In [97]:
datos_adquisidor.splitlines()[3::2]

['fecha, hora, temperatura exterior,Iglo,Ibeam,Idiff',
 '20/12/98,5:00 am, 34.2,0,0,0',
 '20/12/98,5:00 am, 34.2,0,0,0',
 '20/12/98,5:00 am, 34.2,0,0,0',
 '20/12/98,5:00 am, 34.2,0,0,0']